# **Transform Sprints Data**
  1. Read bronze sprints table
  2. Keep only the columns required for analytics (Drop url column)
  3. Standardise Column names using snake_case (constructorid -> constructor_id,driverid -> driver_id, raceName ->race_Name, positionText -> finish_position_text)
  4. Rename columns to make them more meaningful (date -> race_date, grid -> grid_position, laps -> completed_laps, number -> car_number, position -> finish_position)
  5. Filter out rows where season, round, constructor_id or driver_id is null(business key validation)
  6. Remove duplicate records 
  7. Transform values of columns race_name to Title Case 
  8. Write transformated data to Results table 

### Entity Relationship Diagram - Forumula1 Bronze Schema
![](/Workspace/Users/uppalapatiususp@gmail.com/Pavan_Azure_DataEngineer_Projects/Formula1_Project/03-silver/EntityRelation_Diagram.png)




In [0]:
%run ../00-common/01_Environment-Config

In [0]:
bronze_table_nm = f"{catalog_name}.{bronze_schema}.sprints"
silver_table_nm = f"{catalog_name}.{silver_schema}.sprints"

### Step 1,2,3,4 - Read bronze Results table, Select only the requreid columns and standardise column name

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df = (
    spark.read.table(bronze_table_nm)
    .select(
            F.col("date"),
            F.col("raceName"),
            F.col("round"),
            F.col("season"),
            F.col("constructorId"),
            F.col("driverId"),
            F.col("grid"),
            F.col("laps"),
            F.col("number"),
            F.col("points"),            
            F.col("position"),
            F.col("positionText"), 
            F.col("status"),
            F.col("ingestion_timestamp"),
            F.col("source_file")
            )
    .withColumnsRenamed(
        {
            "constructorId": "constructor_Id",
            "driverId": "driver_Id",
            "raceName": "race_Name",
            "date": "race_date",            
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "final_position",
            "positionText": "final_position_text"
        }
    )
    
)

In [0]:
display(sprints_df)

### Step - 5 & 6  Apply Data Quality Checks
- Filter our rows where season, round, constructor_id or driver_id is null (Business Key Validation)
- Remove Duplicate Records 

In [0]:
sprints_valid_df = (
    sprints_df.filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("constructor_Id").isNotNull() &
        F.col("driver_Id").isNotNull()
    )
    .dropDuplicates(["season","round","driver_Id","constructor_Id"])
)


In [0]:
display(sprints_df.count() - sprints_valid_df.count())

### Step-7 Transform values of column race_name to Title Case 

In [0]:
sprints_final_df = (
    sprints_valid_df.withColumn("race_Name",
                                F.initcap(F.col("race_Name"))
                                )
                    )


### Step - 8. Write transformated data to Results table 

In [0]:
(
    sprints_final_df
        .write.mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table_nm)
)

In [0]:
spark.table(silver_table_nm).display()